# Home Credit Risk Analysis

Business-style credit risk analysis notebook modeled after a Lending Club style portfolio review.

Focus:
- portfolio snapshot
- default-rate ladders across core risk variables
- borrower segment analysis
- data quality and missingness
- auxiliary-table coverage
- candidate risk drivers for modeling

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "dataset").exists() and (ROOT.parent / "dataset").exists():
    ROOT = ROOT.parent
if not (ROOT / "dataset").exists():
    raise RuntimeError("Run this notebook from the repo root or from notebooks/.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATASET_DIR = ROOT / "dataset"
OUTPUTS_DIR = ROOT / "outputs"
ROOT

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
plt.style.use("ggplot")

def safe_divide(a, b):
    b = b.replace({0: np.nan})
    return a / b

In [ ]:
application_train = pd.read_csv(DATASET_DIR / "application_train.csv")
application_test = pd.read_csv(DATASET_DIR / "application_test.csv")

def add_risk_features(df):
    frame = df.copy()
    frame["DAYS_EMPLOYED_ANOM"] = (frame["DAYS_EMPLOYED"] == 365243).astype(int)
    frame.loc[frame["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan
    frame["payment_rate"] = safe_divide(frame["AMT_ANNUITY"], frame["AMT_CREDIT"])
    frame["credit_income_ratio"] = safe_divide(frame["AMT_CREDIT"], frame["AMT_INCOME_TOTAL"])
    frame["annuity_income_pct"] = safe_divide(frame["AMT_ANNUITY"], frame["AMT_INCOME_TOTAL"])
    frame["credit_annuity_ratio"] = safe_divide(frame["AMT_CREDIT"], frame["AMT_ANNUITY"])
    frame["credit_goods_ratio"] = safe_divide(frame["AMT_CREDIT"], frame["AMT_GOODS_PRICE"])
    frame["income_per_person"] = safe_divide(frame["AMT_INCOME_TOTAL"], frame["CNT_FAM_MEMBERS"])
    frame["age_years"] = -frame["DAYS_BIRTH"] / 365.0
    frame["employment_years"] = -frame["DAYS_EMPLOYED"] / 365.0
    ext = frame[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    frame["ext_source_mean"] = ext.mean(axis=1)
    frame["ext_source_min"] = ext.min(axis=1)
    frame["ext_source_max"] = ext.max(axis=1)
    frame["ext_source_count"] = ext.notna().sum(axis=1)
    return frame

train = add_risk_features(application_train)
test = add_risk_features(application_test)

master_path = OUTPUTS_DIR / "features" / "master_table.pkl"
master = pd.read_pickle(master_path) if master_path.exists() else None
print("application_train shape:", train.shape)
print("application_test shape:", test.shape)
print("master_table available:", master is not None)

## Portfolio Snapshot

In [ ]:
snapshot = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train),
            "customers": train["SK_ID_CURR"].nunique(),
            "default_rate": train["TARGET"].mean(),
            "avg_income": train["AMT_INCOME_TOTAL"].mean(),
            "avg_credit": train["AMT_CREDIT"].mean(),
            "avg_annuity": train["AMT_ANNUITY"].mean(),
        },
        {
            "dataset": "test",
            "rows": len(test),
            "customers": test["SK_ID_CURR"].nunique(),
            "default_rate": np.nan,
            "avg_income": test["AMT_INCOME_TOTAL"].mean(),
            "avg_credit": test["AMT_CREDIT"].mean(),
            "avg_annuity": test["AMT_ANNUITY"].mean(),
        },
    ]
)
snapshot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
train["TARGET"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["#4C78A8", "#E45756"])
axes[0].set_title("Target Distribution")
axes[0].set_xlabel("TARGET")
axes[0].set_ylabel("Count")

snapshot.set_index("dataset")[["avg_income", "avg_credit", "avg_annuity"]].plot(kind="bar", ax=axes[1])
axes[1].set_title("Train vs Test Averages")
axes[1].set_ylabel("Amount")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## Risk Ladders

In [ ]:
def make_risk_ladder(df, feature, bins=5):
    sample = df[[feature, "TARGET"]].dropna().copy()
    sample["bucket"] = pd.qcut(sample[feature], q=bins, duplicates="drop")
    grouped = (
        sample.groupby("bucket", observed=True)["TARGET"]
        .agg(default_rate="mean", count="size")
        .reset_index()
    )
    grouped["feature"] = feature
    grouped["bucket"] = grouped["bucket"].astype(str)
    return grouped[["feature", "bucket", "count", "default_rate"]]

ladder_features = [
    "ext_source_mean",
    "payment_rate",
    "credit_income_ratio",
    "credit_annuity_ratio",
    "age_years",
]

risk_ladders = pd.concat([make_risk_ladder(train, feature) for feature in ladder_features], ignore_index=True)
risk_ladders

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.ravel()
for ax, feature in zip(axes, ladder_features, strict=False):
    plot_df = risk_ladders[risk_ladders["feature"] == feature]
    ax.plot(range(len(plot_df)), plot_df["default_rate"], marker="o", color="#E45756")
    ax.set_title(feature)
    ax.set_xticks(range(len(plot_df)))
    ax.set_xticklabels(plot_df["bucket"], rotation=45, ha="right")
    ax.set_ylabel("Default Rate")
for idx in range(len(ladder_features), len(axes)):
    axes[idx].axis("off")
plt.tight_layout()
plt.show()

## Borrower Segment Analysis

In [ ]:
segment_features = [
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
]

segment_rows = []
for feature in segment_features:
    grouped = (
        train.assign(level=train[feature].fillna("<MISSING>"))
        .groupby("level", observed=True)["TARGET"]
        .agg(default_rate="mean", count="size")
        .reset_index()
        .sort_values(["default_rate", "count"], ascending=[False, False])
    )
    grouped["feature"] = feature
    segment_rows.append(grouped[["feature", "level", "count", "default_rate"]])

segment_analysis = pd.concat(segment_rows, ignore_index=True)
segment_analysis.head(30)

In [ ]:
plot_df = segment_analysis[segment_analysis["count"] >= 500].copy()
plot_df["label"] = plot_df["feature"] + " | " + plot_df["level"].astype(str)
plot_df = plot_df.sort_values("default_rate", ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(plot_df["label"], plot_df["default_rate"], color="#F58518")
ax.set_title("Highest Default-Rate Borrower Segments (count >= 500)")
ax.set_xlabel("Default Rate")
plt.tight_layout()
plt.show()

## Data Quality

In [ ]:
missing = (
    train.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "feature"})
)

quality_summary = {
    "days_employed_anomalies": int((application_train["DAYS_EMPLOYED"] == 365243).sum()),
    "ext_source_1_missing": float(train["EXT_SOURCE_1"].isna().mean()),
    "ext_source_2_missing": float(train["EXT_SOURCE_2"].isna().mean()),
    "ext_source_3_missing": float(train["EXT_SOURCE_3"].isna().mean()),
}

display(pd.DataFrame([quality_summary]))
missing.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top_missing = missing.head(15).iloc[::-1]
ax.barh(top_missing["feature"], top_missing["missing_fraction"], color="#72B7B2")
ax.set_title("Top Missing Features")
ax.set_xlabel("Missing Fraction")
plt.tight_layout()
plt.show()

## Auxiliary Coverage

In [ ]:
coverage_specs = {
    "bureau": ("bureau.csv", "SK_ID_CURR"),
    "previous_application": ("previous_application.csv", "SK_ID_CURR"),
    "pos_cash_balance": ("POS_CASH_balance.csv", "SK_ID_CURR"),
    "credit_card_balance": ("credit_card_balance.csv", "SK_ID_CURR"),
    "installments_payments": ("installments_payments.csv", "SK_ID_CURR"),
}

train_ids = set(train["SK_ID_CURR"])
coverage_rows = []
for table_name, (filename, key) in coverage_specs.items():
    ids = pd.read_csv(DATASET_DIR / filename, usecols=[key])[key].dropna().unique()
    matched = sum(1 for value in ids if value in train_ids)
    coverage_rows.append(
        {
            "table": table_name,
            "unique_ids": len(ids),
            "matched_train_ids": matched,
            "share_of_train_ids": matched / train["SK_ID_CURR"].nunique(),
        }
    )

coverage = pd.DataFrame(coverage_rows).sort_values("share_of_train_ids", ascending=False)
coverage

## Candidate Risk Drivers

In [ ]:
candidate_numeric = [
    "ext_source_mean",
    "ext_source_min",
    "ext_source_max",
    "payment_rate",
    "credit_income_ratio",
    "annuity_income_pct",
    "credit_annuity_ratio",
    "credit_goods_ratio",
    "income_per_person",
    "age_years",
    "employment_years",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

if master is not None:
    master_train = master[master["TARGET"].notna()].copy()
    master_train["TARGET"] = master_train["TARGET"].astype(int)
    for feature in ["bureau_total_debt_ratio", "install_window_365d_PAYMENT_PCT_mean", "previous_refused_count"]:
        if feature in master_train.columns:
            candidate_numeric.append(feature)
    driver_frame = master_train
else:
    driver_frame = train

rows = []
for feature in dict.fromkeys(candidate_numeric):
    if feature not in driver_frame.columns:
        continue
    series = driver_frame[feature]
    if series.nunique(dropna=True) < 10:
        continue
    filled = series.fillna(series.median())
    auc = roc_auc_score(driver_frame["TARGET"], filled)
    rows.append({"feature": feature, "univariate_auc": max(auc, 1 - auc)})

drivers = pd.DataFrame(rows).sort_values("univariate_auc", ascending=False)
drivers.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = drivers.head(12).iloc[::-1]
ax.barh(plot_df["feature"], plot_df["univariate_auc"], color="#4C78A8")
ax.set_title("Top Candidate Risk Drivers by Univariate AUC")
ax.set_xlabel("Univariate AUC")
plt.tight_layout()
plt.show()

## Analyst Notes

In [ ]:
notes = {
    "portfolio_default_rate": float(train["TARGET"].mean()),
    "strongest_application_signals": drivers.head(5)[["feature", "univariate_auc"]].to_dict(orient="records"),
    "high_risk_segment_examples": segment_analysis.sort_values(["default_rate", "count"], ascending=[False, False]).head(10).to_dict(orient="records"),
    "data_quality_flags": quality_summary,
}
notes